# Notebook 05: LLM API Basics

> **Goal:** Learn to call LLM APIs (OpenAI + Anthropic) like a professional — with proper error handling, cost tracking, streaming, and structured outputs.

**What you'll learn:**
1. OpenAI and Anthropic API fundamentals
2. Temperature, tokens, and context windows
3. Streaming responses
4. Structured JSON output with Pydantic
5. Cost tracking and optimization
6. Building a multi-turn conversation

**Time:** ~3 hours

**Prerequisites:** Set up API keys as environment variables:
```bash
export OPENAI_API_KEY='sk-...'
export ANTHROPIC_API_KEY='sk-ant-...'
```

In [ ]:
# Install required packages (uncomment if needed)
# !pip install openai anthropic pydantic tiktoken

import os
import json
import time
from openai import OpenAI
from anthropic import Anthropic
from pydantic import BaseModel, Field
from typing import Optional
import tiktoken

# Initialize clients
openai_client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))
anthropic_client = Anthropic(api_key=os.environ.get('ANTHROPIC_API_KEY'))

print('Clients initialized!')
print('Note: Set OPENAI_API_KEY and ANTHROPIC_API_KEY env vars before running.')

## Part 1: Understanding Tokens

In [ ]:
# Tokens are NOT words — they're sub-word units
# Rule of thumb: 1 token ≈ 0.75 words ≈ 4 characters

def count_tokens(text: str, model: str = 'gpt-4o') -> int:
    """Count tokens using tiktoken (same tokenizer OpenAI uses)."""
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))

examples = [
    'Hello world',
    'The quick brown fox jumps over the lazy dog.',
    'Machine learning is a subset of artificial intelligence.',
    'GPT-4 uses transformer architecture with attention mechanisms.',
    'Supercalifragilisticexpialidocious',
    '你好世界',  # Chinese: Hello World
]

print(f'{"Text":<50} {"Chars":>6} {"Words":>6} {"Tokens":>7}')
print('-' * 72)
for text in examples:
    tokens = count_tokens(text)
    words = len(text.split())
    print(f'{text:<50} {len(text):>6} {words:>6} {tokens:>7}')

In [ ]:
# Cost estimation
COSTS_PER_1M_TOKENS = {
    'gpt-4o':           {'input': 5.00,  'output': 15.00},
    'gpt-4o-mini':      {'input': 0.15,  'output': 0.60},
    'claude-opus-4-6': {'input': 15.00, 'output': 75.00},
    'claude-haiku-4-5': {'input': 0.80,  'output': 4.00},
}

def estimate_cost(input_tokens: int, output_tokens: int, model: str) -> float:
    costs = COSTS_PER_1M_TOKENS.get(model, {'input': 5.0, 'output': 15.0})
    return (input_tokens * costs['input'] + output_tokens * costs['output']) / 1_000_000

print('COST COMPARISON (per 1000 requests: 500 input + 200 output tokens)')
print('=' * 55)
for model in COSTS_PER_1M_TOKENS:
    cost_per_1k = estimate_cost(500, 200, model) * 1000
    print(f'{model:<25}: ${cost_per_1k:.4f} per 1K requests')

print()
print('💡 Use gpt-4o-mini or claude-haiku for classification/extraction tasks')
print('💡 Reserve GPT-4o/claude-opus for complex reasoning tasks')

## Part 2: OpenAI API — Core Patterns

In [ ]:
# Basic completion
response = openai_client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant. Be concise.'},
        {'role': 'user', 'content': 'What is the difference between a list and a tuple in Python?'},
    ],
    temperature=0,         # 0 = deterministic; 1.0 = more creative
    max_tokens=300,        # Limit output length
)

print('=== RESPONSE ===')
print(response.choices[0].message.content)
print()
print('=== METADATA ===')
print(f'Model: {response.model}')
print(f'Input tokens:  {response.usage.prompt_tokens}')
print(f'Output tokens: {response.usage.completion_tokens}')
cost = estimate_cost(response.usage.prompt_tokens,
                     response.usage.completion_tokens, 'gpt-4o-mini')
print(f'Cost: ${cost:.6f}')

In [ ]:
# Temperature effect — run this to see the difference
prompt = 'Give me a creative product name for an AI-powered coffee machine.'

print('TEMPERATURE EFFECT')
print('=' * 50)
for temp in [0.0, 0.5, 1.0, 1.5]:
    response = openai_client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=temp,
        max_tokens=50,
    )
    print(f'temp={temp}: {response.choices[0].message.content.strip()}')

In [ ]:
# Streaming response — see tokens as they're generated
print('STREAMING RESPONSE:')
print('=' * 50)

stream = openai_client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{
        'role': 'user',
        'content': 'Explain gradient descent in 3 bullet points.'
    }],
    stream=True,  # Enable streaming
    max_tokens=200,
)

full_response = ''
for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        token = chunk.choices[0].delta.content
        full_response += token
        print(token, end='', flush=True)  # Print as received

print()  # Newline after streaming

## Part 3: Structured JSON Output with Pydantic

In [ ]:
# Define your output schema with Pydantic
class JobPosting(BaseModel):
    job_title: str = Field(description='Exact job title')
    company: str = Field(description='Company name')
    location: str = Field(description='City, State or Remote')
    is_remote: bool = Field(description='Whether the job is remote')
    required_skills: list[str] = Field(description='List of required technical skills')
    years_experience: Optional[int] = Field(None, description='Years of experience required')
    salary_min: Optional[int] = Field(None, description='Minimum salary in USD')
    salary_max: Optional[int] = Field(None, description='Maximum salary in USD')
    seniority: str = Field(description='Junior, Mid, Senior, Staff, or Principal')


# Sample job posting to parse
job_text = """
Senior ML Engineer at DeepMind

We're looking for an experienced Machine Learning Engineer to join our research team
in London (hybrid: 3 days onsite). You'll need 5+ years of experience with PyTorch,
distributed training, and MLOps tools. Experience with Kubernetes and Ray is a plus.
Python expertise is required. Compensation: $180,000 - $240,000 USD depending on experience.
"""

# Use OpenAI with JSON mode + Pydantic for guaranteed structured output
response = openai_client.chat.completions.create(
    model='gpt-4o-mini',
    response_format={'type': 'json_object'},
    messages=[
        {
            'role': 'system',
            'content': f'''Extract job posting information and return it as JSON.
Use this exact schema: {JobPosting.model_json_schema()}'''
        },
        {'role': 'user', 'content': job_text}
    ],
    temperature=0,
)

# Parse and validate with Pydantic
raw_json = json.loads(response.choices[0].message.content)
job = JobPosting(**raw_json)

print('PARSED JOB POSTING')
print('=' * 40)
print(f'Title:       {job.job_title}')
print(f'Company:     {job.company}')
print(f'Location:    {job.location}')
print(f'Remote:      {job.is_remote}')
print(f'Seniority:   {job.seniority}')
print(f'Experience:  {job.years_experience}+ years')
print(f'Salary:      ${job.salary_min:,} - ${job.salary_max:,}')
print(f'Skills:      {", ".join(job.required_skills)}')

## Part 4: Anthropic Claude API

In [ ]:
# Anthropic API — similar structure to OpenAI but with differences

# Key differences:
# - System prompt is a separate parameter (not a message)
# - Model names: claude-opus-4-6, claude-haiku-4-5-20251001
# - No 'json_object' response format — prompt for JSON explicitly

response = anthropic_client.messages.create(
    model='claude-haiku-4-5-20251001',  # Fast and cheap — use for simple tasks
    max_tokens=500,
    system='You are an expert Python teacher. Explain concepts clearly with examples.',
    messages=[
        {'role': 'user', 'content': 'Explain Python decorators with a practical example.'}
    ],
)

print('CLAUDE RESPONSE:')
print(response.content[0].text)
print()
print(f'Input tokens:  {response.usage.input_tokens}')
print(f'Output tokens: {response.usage.output_tokens}')

In [ ]:
# Claude's vision capability — analyze images
import base64
import urllib.request

# Download a sample image (matplotlib chart)
# In real use, you'd load your own image
try:
    # Use a chart we saved earlier if it exists
    with open('visualization_gallery.png', 'rb') as f:
        image_data = base64.standard_b64encode(f.read()).decode('utf-8')

    response = anthropic_client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=300,
        messages=[{
            'role': 'user',
            'content': [
                {
                    'type': 'image',
                    'source': {
                        'type': 'base64',
                        'media_type': 'image/png',
                        'data': image_data,
                    }
                },
                {
                    'type': 'text',
                    'text': 'Describe what you see in this chart. What are the key insights?'
                }
            ]
        }]
    )
    print('VISION ANALYSIS:')
    print(response.content[0].text)
except FileNotFoundError:
    print('Run notebook 03 first to generate the visualization_gallery.png file.')
    print('Or replace with any image file you have.')

## Part 5: Multi-Turn Conversation Manager

In [ ]:
class ConversationManager:
    """
    Manages a multi-turn conversation with context window awareness.
    Automatically truncates old messages to stay within token limits.
    """

    def __init__(self, system_prompt: str, model: str = 'gpt-4o-mini',
                 max_context_tokens: int = 8000):
        self.model = model
        self.system = system_prompt
        self.max_tokens = max_context_tokens
        self.messages = []
        self.total_cost = 0.0
        self.client = OpenAI()

    def _count_tokens(self, messages: list) -> int:
        enc = tiktoken.encoding_for_model(self.model)
        return sum(len(enc.encode(m['content'])) for m in messages)

    def _trim_context(self):
        """Remove oldest messages to stay within token limit."""
        while (self._count_tokens(self.messages) > self.max_tokens
               and len(self.messages) > 2):
            self.messages.pop(0)  # Remove oldest user/assistant pair
            if self.messages and self.messages[0]['role'] == 'assistant':
                self.messages.pop(0)

    def chat(self, user_message: str) -> str:
        """Send a message and get a response."""
        self.messages.append({'role': 'user', 'content': user_message})
        self._trim_context()

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{'role': 'system', 'content': self.system}] + self.messages,
            temperature=0.7,
        )

        assistant_message = response.choices[0].message.content
        self.messages.append({'role': 'assistant', 'content': assistant_message})

        cost = estimate_cost(response.usage.prompt_tokens,
                             response.usage.completion_tokens, self.model)
        self.total_cost += cost

        return assistant_message

    def get_stats(self) -> dict:
        return {
            'turns': len(self.messages) // 2,
            'context_tokens': self._count_tokens(self.messages),
            'total_cost_usd': round(self.total_cost, 6),
        }


# Demo: Python tutor conversation
tutor = ConversationManager(
    system_prompt='You are a Python tutor. Give concise, practical answers with code examples.',
    model='gpt-4o-mini'
)

conversations = [
    'What is a list comprehension?',
    'Show me a more complex example with if condition.',
    'Can I use dictionary comprehensions the same way?',
]

for message in conversations:
    print(f'\n👤 USER: {message}')
    response = tutor.chat(message)
    print(f'🤖 ASSISTANT: {response[:400]}...' if len(response) > 400 else f'🤖 ASSISTANT: {response}')

print('\n📊 SESSION STATS:')
print(tutor.get_stats())

## Summary

| Task | Best Model | Temperature |
|------|------------|-------------|
| Classification, extraction | gpt-4o-mini / claude-haiku | 0 |
| Summarization | gpt-4o-mini / claude-haiku | 0.3 |
| Code generation | gpt-4o / claude-sonnet | 0 |
| Creative writing | gpt-4o / claude-opus | 0.8-1.0 |
| Complex reasoning | gpt-4o / claude-opus | 0.2-0.5 |

## Challenge
1. Build a `ResumeAnalyzer` class that takes a job description and resume text, extracts key requirements using Pydantic schema, and scores the match 0-100
2. Add retry logic to the `ConversationManager` for handling rate limits
3. Compare GPT-4o-mini vs Claude Haiku on 10 classification tasks for accuracy and cost